# PGM end-to-end pipeline

This notebook runs **every stage** of the project in order: load raw data, preprocess, optionally explore QC plots, cluster cells, fit three Gaussian graphical model (GGM) variants, evaluate them, and record a reproducibility snapshot.

**Model sketch:** Graphical Lasso minimizes \(-\log\det\Theta + \mathrm{tr}(S\Theta) + \alpha\|\Theta\|_1\) with \(\Theta \succeq 0\). For mixture states, a weighted scatter \(S^{(k)}\) is used; the KG-biased stage blends covariance with a prior \(S' = S^{(k)} + \lambda P\) (see `src/pgm/models/`).

Artifacts land under `data/`, `reports/`, and `logs/` as configured in `configs/default.yaml`, optional presets such as `configs/quick.yaml`, and `smoke.yaml` when smoke mode is on.

## 1. Setup

The package expects to run with the **repository root** on `sys.path` and as the working directory so paths like `configs/`, `data/`, and `datasets/` resolve correctly. This cell moves to the parent of `notebooks/` and imports the pipeline entrypoints.

For **tiny** fast runs, use `SMOKE=1` or `smoke=True` (merges `configs/smoke.yaml`). For a **medium** runtime (about 1.6k cells, fewer HVGs/bootstraps/STRING genes), use `preset="quick"` or set env `PGM_PRESET=quick` (merges `configs/quick.yaml` on top of default).

For **tuned** full-cell runs (HVG sweep + promoted winner, isolated checkpoints under `data/checkpoints/tuning/...`), use `preset="tuning_best"` or the other `configs/tuning_*.yaml` presets; use `python scripts/tuning_sweep.py` to regenerate sweep tables and refresh `tuning_best.yaml`.

In [1]:
from pathlib import Path
import json
import os
import time

os.chdir(Path("..").resolve())

FORCE_RERUN = True  # set True to ignore existing checkpoints / outputs where supported

from pgm.config.loader import load_config
from pgm.pipelines.ingest import ingest_pbmc_pipeline
from pgm.pipelines.preprocess import run_preprocess_pipeline
from pgm.pipelines.eda import run_eda
from pgm.pipelines.cluster import run_clustering_pipeline
from pgm.pipelines.global_ggm import run_global_ggm_pipeline
from pgm.pipelines.soft_ggm import run_soft_weighted_ggm_pipeline
from pgm.pipelines.kg_pipeline import run_kg_pipeline
from pgm.pipelines.evaluation import run_evaluation_pipeline
from pgm.utils.env import snapshot_run_environment


Loads `configs/default.yaml`, optionally merges `configs/{preset}.yaml` when `preset` is set (or `PGM_PRESET` is set in the environment), then merges `configs/smoke.yaml` when smoke mode is enabled. Validates into a `ProjectConfig`. All later stages read this single object so behavior is consistent.

In [2]:
# preset=None → default.yaml full PBMC. preset="quick" → mid-scale. preset="tuning_best" → sweep-promoted hyperparameters.
# smoke=True for tiny runs (do not combine with tuning presets — smoke overrides HVG counts).
# Graphical Lasso backend: "sklearn" or "gglasso"; extra_overrides wins over YAML.
GL_BACKEND = "gglasso"
cfg = load_config(
    smoke=False,
    preset="tuning_best",
    extra_overrides={"models": {"gl_backend": GL_BACKEND}},
)

## 3. Data ingestion

Reads the 10x PBMC matrix from `data.tenx_mtx_dir`, performs basic validation, writes a JSON summary under `reports/results/`, and saves interim AnnData to `data/interim/` (`ingestion.write_interim_filename`). Checkpoints avoid recomputation when `FORCE_RERUN` is False.


In [3]:
t_ingest = time.perf_counter()
interim_raw_h5ad = ingest_pbmc_pipeline(cfg, force=FORCE_RERUN)
print(f"Ingestion → {interim_raw_h5ad} ({time.perf_counter() - t_ingest:.2f}s)")


Ingestion → F:\Research\PGM\data\interim\pbmc3k_raw.h5ad (0.32s)


## 4. Preprocessing

Standard single-cell workflow: QC filters (genes/cells, mitochondrial fraction), normalization to a target sum, log transform, highly variable genes, scaling, PCA. Writes `data/processed/` AnnData including `X_pca` for downstream neighbors and mixture models.


In [4]:
t_pre = time.perf_counter()
processed_h5ad = run_preprocess_pipeline(cfg, interim_raw_h5ad, force=FORCE_RERUN)
print(f"Preprocess → {processed_h5ad} ({time.perf_counter() - t_pre:.2f}s)")


2026-05-15 16:10:41 pgm.preprocessing INFO Running PCA n_comps=35 (nhvg=300)
2026-05-15 16:10:41 pgm.preprocessing INFO Preprocessed shape=(2698, 300); HVG count=300
2026-05-15 16:10:41 pgm.checkpoint INFO Saved AnnData checkpoint F:\Research\PGM\data\checkpoints\tuning\best\preprocess_done.h5ad (AnnData)
2026-05-15 16:10:41 pgm.pipelines.preprocess INFO Preprocessed 1.12s → F:\Research\PGM\data\processed\pbmc3k_tuning_best.h5ad shape=(2698, 300)
Preprocess → F:\Research\PGM\data\processed\pbmc3k_tuning_best.h5ad (1.12s)


### 4b. Processed matrix sanity check

Loads the written processed `.h5ad` and prints non-finite counts, value range, and how many genes have ~zero standard deviation across cells. NaNs after scaling usually mean zero-variance genes; Graphical Lasso is sensitive to singular covariance. Preprocessing now drops those genes before PCA.

In [5]:
import scanpy as sc

from pgm.utils.adata_audit import format_matrix_summary, summarize_adata_matrix

_qc = sc.read_h5ad(processed_h5ad)
print(format_matrix_summary(summarize_adata_matrix(_qc)))
print("\nobs columns (sample):", list(_qc.obs.columns[:12]))
_qc.obs.head()

X shape: 2698 × 300
non-finite values: 0 (nan=0, +inf=0, -inf=0)
finite min/mean/max: -0.971822 / -0.0062506 / 10
genes with ~zero std (ddof=1): 0; genes with non-finite std: 0

obs columns (sample): ['n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'n_genes']


,n_genes_by_counts,log1p_n_genes_by_counts,total_counts,log1p_total_counts,pct_counts_in_top_50_genes,pct_counts_in_top_100_genes,pct_counts_in_top_200_genes,pct_counts_in_top_500_genes,total_counts_mt,log1p_total_counts_mt,pct_counts_mt,n_genes
AAACATACAACCAC-1,781,6.661855,2421.0,7.792349,47.748864,63.279637,74.969021,88.393226,73.0,4.304065,3.015283,781
AAACATTGAGCTAC-1,1352,7.210080,4903.0,8.497807,45.502753,61.023863,71.813176,82.622884,186.0,5.231109,3.793596,1352
AAACATTGATCAGC-1,1131,7.031741,3149.0,8.055158,41.314703,53.794856,65.449349,79.961893,28.0,3.367296,0.889171,1131
AAACCGTGCTTCCG-1,960,6.867974,2639.0,7.878534,39.029936,52.898825,66.691929,82.569155,46.0,3.850147,1.743085,960
AAACCGTGTATGCG-1,522,6.259581,981.0,6.889591,44.852192,55.657492,67.176351,97.757390,12.0,2.564949,1.223242,522


## 5. Exploratory data analysis (optional)

Builds QC-style figures under `reports/figures/eda/` and a short markdown summary under `reports/eda/`. This stage is **not** required for the modeling chain but helps verify preprocessing; it uses the processed AnnData path and respects smoke sampling hints from config.


In [ ]:
t_eda = time.perf_counter()
eda_summary_path = run_eda(cfg, processed_h5ad, force=FORCE_RERUN)
print(f"EDA summary → {eda_summary_path} ({time.perf_counter() - t_eda:.2f}s)")


2026-05-15 16:10:42 pgm.pipelines.eda INFO EDA: using existing X_pca from processed AnnData (skip re-normalize/log1p)


## 6. Clustering and latent visualization

Computes nearest neighbors on PCA, Leiden clustering, UMAP embeddings, and fits a Gaussian mixture on the configured PCA representation to obtain **soft** component assignments. Saves clustered AnnData (default sibling `*_clustered.h5ad` in `data/processed/`) and Leiden/GMM diagnostic figures under `reports/figures/clustering/`.


In [ ]:
t_cl = time.perf_counter()
clustered_h5ad = run_clustering_pipeline(cfg, processed_h5ad, force=FORCE_RERUN)
print(f"Clustering → {clustered_h5ad} ({time.perf_counter() - t_cl:.2f}s)")


2026-05-15 15:46:40 pgm.clustering INFO GMM fit done; mean soft assignment entropy 0.0414
2026-05-15 15:46:42 pgm.checkpoint INFO Saved AnnData checkpoint F:\Research\PGM\data\checkpoints\cluster_done.h5ad (AnnData)
2026-05-15 15:46:42 pgm.pipelines.cluster INFO Clustering done 6.61s → F:\Research\PGM\data\processed\pbmc3k_processed_clustered.h5ad (n_obs=2698)
Clustering → F:\Research\PGM\data\processed\pbmc3k_processed_clustered.h5ad (6.62s)


## 7. Global graphical Lasso (one graph for all cells)

Fits a **single** sparse inverse covariance (precision) matrix and adjacency from the full processed gene set (subject to model code paths). Serialized bundle under `data/checkpoints/global_ggm.joblib` for evaluation and comparison.


In [ ]:
t_gg = time.perf_counter()
global_ggm_joblib = run_global_ggm_pipeline(cfg, clustered_h5ad, force=FORCE_RERUN)
print(f"Global GGM → {global_ggm_joblib} ({time.perf_counter() - t_gg:.2f}s)")


2026-05-15 15:46:42 pgm.pipelines.global_ggm INFO Global GGM loading AnnData F:\Research\PGM\data\processed\pbmc3k_processed_clustered.h5ad
2026-05-15 15:46:42 pgm.pipelines.global_ggm INFO Global GGM AnnData read 0.057s shape=(2698, 2000)
2026-05-15 15:46:42 pgm.models.glm INFO expression_dense shape=(2698, 2000) storage=dense → float64 0.036s (41.2 MiB)
2026-05-15 15:46:42 pgm.models.global_ggm INFO Global GGM fitting on X shape (2698, 2000) alpha=0.25
2026-05-15 15:46:42 pgm.models.glm INFO [global_ggm] empirical_covariance n=2698 p=2000 0.105s
2026-05-15 15:46:42 pgm.models.glm INFO [global_ggm] covariance_target_shrink s=0.1000 tr_before=1.6446e+03 mu_tr_over_p=8.2232e-01
2026-05-15 15:46:44 pgm.models.glm INFO [global_ggm] stabilize_cov p=2000 eig_min_pre_floor=8.2664e-02 ridge_cfg=1.0000e-04 floor_applied=1.0000e-04 (extra 0.0000e+00) eigvalsh=0.800s total=2.299s tr_after_floor=1.6448e+03
2026-05-15 15:46:44 pgm.models.glm INFO [global_ggm] graphical_lasso_from_covariance begin 

## 8. Soft mixture–weighted GGM (per latent state)

Uses GMM soft labels to fit **one graphical model per mixture component**, weighting cells by responsibility. Saves `data/checkpoints/soft_weighted_ggm.joblib` and component-level sparsity figures under `reports/figures/ggm/`. Compares heterogeneous structure across states.


In [ ]:
print(
    "[Section 8] Starting soft mixture–weighted GGM (one Graphical Lasso per GMM state).\n"
    f"  HVGs (config): n_top_hvg={cfg.preprocessing.n_top_hvg}\n"
    f"  Mixture: gmm_n_components={cfg.clustering.gmm_n_components}\n"
    f"  Solver: gl_backend={cfg.models.gl_backend!r}  "
    f"gglasso_solver={cfg.models.gglasso_solver!r}\n"
    f"    gl_mode={cfg.models.gl_mode!r} (sklearn CD/LARS only)  "
    f"gl_alpha={cfg.models.gl_alpha}  "
    f"retry_mults={list(cfg.models.gl_alpha_retry_multipliers)}\n"
    f"  Covariance: covariance_ridge={cfg.models.covariance_ridge}  "
    f"covariance_shrinkage={cfg.models.covariance_shrinkage}\n"
    f"  Iterations: gl_max_iter={cfg.models.gl_max_iter}  gl_tol={cfg.models.gl_tol}\n"
    "  Live logs: INFO lines stream below; full detail also in logs/pgm.log.",
    flush=True,
)
t_soft = time.perf_counter()
soft_ggm_joblib = run_soft_weighted_ggm_pipeline(cfg, clustered_h5ad, force=FORCE_RERUN)
print(f"Soft GGM → {soft_ggm_joblib} ({time.perf_counter() - t_soft:.2f}s)")


[Section 8] Starting soft mixture–weighted GGM (one Graphical Lasso per GMM state).
  HVGs (config): n_top_hvg=2000
  Mixture: gmm_n_components=8
  Solver: gl_backend='gglasso'  gglasso_solver='block_sgl'
    gl_mode='cd' (sklearn CD/LARS only)  gl_alpha=0.25  retry_mults=[1.0, 2.0, 4.0, 8.0]
  Covariance: covariance_ridge=0.0001  covariance_shrinkage=0.1
  Iterations: gl_max_iter=200  gl_tol=0.001
  Live logs: INFO lines stream below; full detail also in logs/pgm.log.
2026-05-15 15:46:46 pgm.pipelines.soft_ggm INFO Soft GGM loading AnnData F:\Research\PGM\data\processed\pbmc3k_processed_clustered.h5ad
2026-05-15 15:46:46 pgm.pipelines.soft_ggm INFO Soft GGM AnnData read 0.049s shape=(2698, 2000) obsm_keys=['X_gmm_proba', 'X_pca', 'X_umap']
2026-05-15 15:46:46 pgm.models.glm INFO expression_dense shape=(2698, 2000) storage=dense → float64 0.037s (41.2 MiB)
[Soft GGM] fit_soft_component_graphs: K=8 matrix=2698x2000 mean_assignment_entropy=0.0414 gl_mode='cd' gl_alpha=0.25 ridge=1.00e-04

## 9. Knowledge-graph prior GGM (STRING)

Queries STRING (cached under `data/external/string_cache/`) for high-confidence edges among HVGs, builds a sparse prior matrix \(P\), and fits KG-biased GGMs per mixture state. Outputs `data/checkpoints/kg_soft_ggm.joblib` and related exports for evaluation against STRING-derived reference pairs.


In [ ]:
t_kg = time.perf_counter()
kg_joblib = run_kg_pipeline(cfg, clustered_h5ad, force=FORCE_RERUN)
print(f"KG GGM → {kg_joblib} ({time.perf_counter() - t_kg:.2f}s)")


2026-05-15 15:47:21 pgm.pipelines.kg INFO KG pipeline loading AnnData F:\Research\PGM\data\processed\pbmc3k_processed_clustered.h5ad
2026-05-15 15:47:21 pgm.pipelines.kg INFO KG AnnData read 0.122s shape=(2698, 2000)
2026-05-15 15:47:21 pgm.kg.string WARNING STRING query truncated 2000 → 500 genes
2026-05-15 15:47:21 pgm.kg.string INFO STRING network request start genes=500 batch_size=400 required_score=700 cache_write=True
2026-05-15 15:47:24 pgm.kg.string INFO STRING batch 0-400 ok
2026-05-15 15:47:25 pgm.kg.string INFO STRING batch 400-500 ok
2026-05-15 15:47:25 pgm.kg.string INFO Wrote STRING cache F:\Research\PGM\data\external\string_cache\aaf299cf0bea4576cb5e344f920dca8d_network.parquet
2026-05-15 15:47:25 pgm.kg.string INFO STRING fetch done 3.378s rows=270 n_batches=2
2026-05-15 15:47:25 pgm.pipelines.kg INFO KG STRING edges 3.380s rows=270 cols=['preferredName_A', 'preferredName_B', 'score']
2026-05-15 15:47:25 pgm.pipelines.kg INFO KG prior matrix 0.019s shape=(2000, 2000) of

## 10. Evaluation

Aggregates **bootstrap stability** of the global adjacency, **STRING precision/recall** for the global graph, **degree/sparsity** summaries, and **cross-state similarity** (Jaccard / Frobenius) for soft and KG adjacencies where applicable. Writes `reports/results/metrics_summary.csv`.


In [ ]:
t_ev = time.perf_counter()
metrics_csv = run_evaluation_pipeline(
    cfg,
    clustered_h5ad,
    global_ggm_joblib,
    soft_ggm_joblib,
    kg_joblib,
    force=FORCE_RERUN,
)
print(f"Metrics → {metrics_csv} ({time.perf_counter() - t_ev:.2f}s)")


2026-05-15 15:48:05 pgm.models.glm INFO expression_dense shape=(2698, 2000) storage=dense → float64 0.038s (41.2 MiB)
2026-05-15 15:48:06 pgm.models.glm INFO covariance_target_shrink s=0.1000 tr_before=1.6358e+03 mu_tr_over_p=8.1791e-01
2026-05-15 15:48:08 pgm.models.glm INFO stabilize_cov p=2000 eig_min_pre_floor=8.1791e-02 ridge_cfg=1.0000e-04 floor_applied=1.0000e-04 (extra 0.0000e+00) eigvalsh=0.887s total=2.387s tr_after_floor=1.6360e+03
2026-05-15 15:48:08 pgm.models.glm INFO graphical_lasso_from_covariance begin p=2000 backend=gglasso mode=cd gglasso_solver='block_sgl' alpha_base=0.25 retry_mults=[1.0, 2.0, 4.0, 8.0] max_iter=500 tol=1.00e-03
2026-05-15 15:48:08 pgm.models.glm INFO graphical_lasso attempt (primary) mult=1 alpha_req=2.500000e-01 alpha_eff=2.500000e-01 mode=n/a
2026-05-15 15:48:08 pgm.models.glm INFO graphical_lasso OK (primary) mult=1 alpha_req=2.500000e-01 alpha_eff=2.500000e-01 n_iter=n/a wall=0.338s tr(cov_or_input)=1.6360e+03 tr(precision)=3.2716e+03 ||precis

## 11. Run environment snapshot

Persists dependency versions and run metadata for reproducibility (location defined by `pgm.utils.env.snapshot_run_environment` and config paths). Use this alongside saved checkpoints for paper methods and debugging.


In [ ]:
t_env = time.perf_counter()
env_json = snapshot_run_environment(cfg, tag="full_pipeline")
print(f"Environment → {env_json} ({time.perf_counter() - t_env:.2f}s)")


Environment → F:\Research\PGM\reports\results\full_pipeline_20260515T095017Z.json (0.87s)


## 12. Summary and exports

Collects all stage paths in one dict (same keys as `run_full_pipeline` in `src/pgm/pipelines/full_pipeline.py`), total wall time, and optionally writes a JSON summary under `reports/results/`.


In [ ]:
summary = {
    "interim_raw_h5ad": interim_raw_h5ad,
    "processed_h5ad": processed_h5ad,
    "eda_summary": eda_summary_path,
    "clustered_h5ad": clustered_h5ad,
    "global_ggm_joblib": global_ggm_joblib,
    "soft_ggm_joblib": soft_ggm_joblib,
    "kg_joblib": kg_joblib,
    "metrics_csv": metrics_csv,
    "env_json": env_json,
}
summary["runtime_s"] = time.perf_counter() - t_ingest

out_json = Path("reports/results/final_pipeline_summary.json")
out_json.parent.mkdir(parents=True, exist_ok=True)
out_json.write_text(
    json.dumps({k: str(v) for k, v in summary.items()}, indent=2),
    encoding="utf-8",
)
print(f"Wrote {out_json}")
summary


Wrote reports\results\final_pipeline_summary.json


{'interim_raw_h5ad': WindowsPath('F:/Research/PGM/data/interim/pbmc3k_raw.h5ad'),
 'processed_h5ad': WindowsPath('F:/Research/PGM/data/processed/pbmc3k_processed.h5ad'),
 'eda_summary': WindowsPath('F:/Research/PGM/reports/eda/summary.md'),
 'clustered_h5ad': WindowsPath('F:/Research/PGM/data/processed/pbmc3k_processed_clustered.h5ad'),
 'global_ggm_joblib': WindowsPath('F:/Research/PGM/data/checkpoints/global_ggm.joblib'),
 'soft_ggm_joblib': WindowsPath('F:/Research/PGM/data/checkpoints/soft_weighted_ggm.joblib'),
 'kg_joblib': WindowsPath('F:/Research/PGM/data/checkpoints/kg_soft_ggm.joblib'),
 'metrics_csv': WindowsPath('F:/Research/PGM/reports/results/metrics_summary.csv'),
 'env_json': WindowsPath('F:/Research/PGM/reports/results/full_pipeline_20260515T095017Z.json'),
 'runtime_s': 246.4152893999999}